# MVP — Análise Exploratória e Pré-processamento de Dados
## Desempenho Escolar no SARESP: Classificação de Escolas por Nível de Proficiência

**Aluno:** Carlos Eduardo Nazareth de Oliveira
**Disciplina:** Análise de Dados e Boas Práticas
**Data:** Abril de 2026

## 1. Motivação e Contexto

O SARESP (Sistema de Avaliação de Rendimento Escolar do Estado de São Paulo) é uma avaliação externa aplicada anualmente pela Secretaria da Educação do Estado de São Paulo (SEDUC-SP) a alunos do Ensino Fundamental e Médio. Os resultados são utilizados para aferir a qualidade do ensino e orientar políticas públicas educacionais em todo o estado.

O dataset utilizado neste trabalho contém **resultados agregados por escola** do SARESP referentes aos anos de 2023, 2024 e 2025, abrangendo as disciplinas de Língua Portuguesa e Matemática para o 2º, 5º e 9º anos do Ensino Fundamental, nas redes estadual e municipal.

A escolha deste dataset é motivada pela minha atuação profissional como Engenheiro de Dados em uma organização do terceiro setor focada na transformação da educação pública em São Paulo. Trabalhar com dados educacionais reais me permite aplicar os conhecimentos da disciplina em um contexto que conheço profundamente, produzindo análises com relevância prática.

Cada registro do dataset representa o resultado de uma escola em uma avaliação específica (combinação de ano, série e componente curricular), contendo a proficiência média e a distribuição percentual dos alunos nos quatro níveis de desempenho definidos pelo SARESP: **Abaixo do Básico**, **Básico**, **Adequado** e **Avançado**.

### Fonte dos dados

Os dados foram extraídos do portal de resultados do SARESP/SEDUC-SP e consolidados em formato CSV. O arquivo está disponível no repositório público do GitHub associado a este trabalho.

## 2. Definição do Problema

### 2.1 Descrição do problema

O objetivo deste trabalho é realizar uma análise exploratória completa dos dados do SARESP e preparar os dados para um futuro modelo de **classificação binária** que determine se uma escola atingiu um nível de desempenho **satisfatório** ou **insatisfatório** em uma dada avaliação.

Definimos como **satisfatório** o cenário em que a soma dos percentuais de alunos nos níveis Adequado e Avançado é **maior ou igual a 50%**. Ou seja, a maioria dos alunos da escola demonstrou desempenho dentro ou acima das expectativas para a série avaliada.

### 2.2 Tipo de problema

Trata-se de um problema de **aprendizado supervisionado**, especificamente de **classificação binária**. A variável-alvo será construída a partir dos dados existentes (feature engineering) e assumirá os valores `1` (satisfatório) e `0` (insatisfatório).

### 2.3 Premissas e hipóteses

- **H1:** Escolas da rede estadual podem apresentar padrões de desempenho diferentes das escolas da rede municipal, possivelmente por diferenças na gestão e nos recursos disponíveis.
- **H2:** O desempenho tende a ser menor nas séries mais avançadas (9º ano) em comparação com o 2º ano, já que as avaliações tornam-se mais complexas.
- **H3:** Escolas com maior número total de alunos avaliados podem apresentar resultados mais estáveis (menor variância), dado o efeito de tamanho amostral.
- **H4:** Pode haver diferença significativa de desempenho entre Língua Portuguesa e Matemática.

### 2.4 Restrições e condições de seleção

- Os dados estão **agregados por escola** (não há microdados individuais por aluno).
- Os anos avaliados são 2023, 2024 e 2025.
- Apenas Ensino Fundamental (2º, 5º e 9º anos) está representado.
- As disciplinas avaliadas são Língua Portuguesa e Matemática.

### 2.5 Dicionário de atributos

| Atributo | Tipo | Descrição |
|---|---|---|
| `CDREDE` | Numérico (int) | Código da diretoria regional de ensino |
| `CODMUN` | Numérico (int) | Código do município (SEDUC) |
| `cod_ibge` | Numérico (int) | Código IBGE do município |
| `MUN` | Categórico | Nome do município |
| `NomeDepBol` | Categórico | Tipo de rede (Estadual ou Municipal) |
| `CODESC` | Numérico (int) | Código da escola |
| `NOMESC` | Categórico | Nome da escola |
| `SERIE_ANO` | Categórico | Série/ano avaliado (2º, 5º ou 9º Ano EF) |
| `ANO` | Numérico (int) | Ano de aplicação da avaliação |
| `COMPONENTE` | Categórico | Componente curricular (LP ou MAT) |
| `PROFIC` | Numérico (string*) | Proficiência média da escola |
| `qtd_abx_basico` | Numérico (int) | Nº de alunos no nível Abaixo do Básico |
| `qtd_basico` | Numérico (int) | Nº de alunos no nível Básico |
| `qtd_adequado` | Numérico (int) | Nº de alunos no nível Adequado |
| `qtd_avancado` | Numérico (int) | Nº de alunos no nível Avançado |
| `total_alunos` | Numérico (int) | Total de alunos avaliados na escola |
| `pct_abx_basico` | Numérico (string*) | % de alunos Abaixo do Básico |
| `pct_basico` | Numérico (string*) | % de alunos no nível Básico |
| `pct_adequado` | Numérico (string*) | % de alunos no nível Adequado |
| `pct_avancado` | Numérico (string*) | % de alunos no nível Avançado |
| `Produto` | Numérico (string*) | Produto: PROFIC × total_alunos |

*\*Colunas armazenadas como string devido ao uso de vírgula como separador decimal e/ou sufixo `%`. Serão convertidas na etapa de pré-processamento.*

## 3. Configuração do Ambiente

Importação das bibliotecas utilizadas ao longo do notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# Configurações globais de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Paleta de cores consistente ao longo do notebook
CORES = {
    'primaria': '#2563EB',
    'secundaria': '#F59E0B',
    'sucesso': '#10B981',
    'alerta': '#EF4444',
    'neutro': '#6B7280',
    'niveis': ['#EF4444', '#F59E0B', '#10B981', '#2563EB']
}

print('Bibliotecas importadas com sucesso.')

## 4. Carga dos Dados

O dataset é carregado diretamente do repositório público no GitHub, garantindo reprodutibilidade. O arquivo utiliza ponto-e-vírgula (`;`) como delimitador e vírgula como separador decimal em algumas colunas.

In [ ]:
# URL do dataset no repositório público do GitHub
URL_DATASET = (
    'https://raw.githubusercontent.com/KaduNazareth/mvp-saresp/main/base.csv'
)

df_raw = pd.read_csv(URL_DATASET, sep=';', encoding='utf-8')

print(f'Dataset carregado com sucesso.')
print(f'Dimensões: {df_raw.shape[0]:,} registros × {df_raw.shape[1]} colunas')

## 5. Análise Exploratória de Dados (EDA)

### 5.1 Visão geral do dataset

Iniciamos verificando a estrutura geral dos dados: tipos, primeiras linhas e dimensões.

In [ ]:
# Primeiras linhas do dataset
df_raw.head(10)

Ao inspecionar as primeiras linhas, notamos imediatamente que diversas colunas que deveriam ser numéricas estão armazenadas como texto (`object`/`str`). As colunas `PROFIC`, `Produto` e as colunas de percentual (`pct_*`) utilizam vírgula como separador decimal, e os percentuais possuem o caractere `%` como sufixo. Essas conversões serão tratadas na etapa de pré-processamento.

In [ ]:
# Tipos de dados e informações gerais
df_raw.info()

In [ ]:
# Verificação de valores nulos
nulos = df_raw.isnull().sum()
print('Contagem de valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Nenhum valor nulo encontrado.')

**Achado importante:** O dataset **não contém valores nulos** em nenhuma coluna. Isso indica que os dados passaram por algum processo de curadoria antes da publicação, o que é esperado para dados oficiais de avaliação educacional, onde cada escola avaliada obrigatoriamente gera resultados completos.

Embora não haja valores faltantes para tratar, precisaremos verificar se existem **valores inconsistentes** (por exemplo, percentuais que não somam 100% ou escolas com zero alunos).

In [ ]:
# Verificação de duplicatas
duplicatas = df_raw.duplicated().sum()
print(f'Registros duplicados: {duplicatas}')

### 5.2 Estatísticas descritivas

Para produzir estatísticas descritivas significativas, precisamos antes converter as colunas numéricas que estão como string. Faremos isso de forma temporária aqui, e a conversão definitiva será realizada na etapa de pré-processamento.

In [ ]:
# Cópia de trabalho para EDA com conversões temporárias
df = df_raw.copy()

# Converter colunas com vírgula decimal
colunas_virgula = ['PROFIC', 'Produto']
for col in colunas_virgula:
    df[col] = df[col].str.replace(',', '.').astype(float)

# Converter colunas de percentual (remover '%' e converter vírgula)
colunas_pct = ['pct_abx_basico', 'pct_basico', 'pct_adequado', 'pct_avancado']
for col in colunas_pct:
    df[col] = df[col].str.replace('%', '').str.replace(',', '.').astype(float)

print('Conversões temporárias realizadas.')
df.dtypes

In [ ]:
# Resumo estatístico das variáveis numéricas
colunas_numericas = [
    'PROFIC', 'qtd_abx_basico', 'qtd_basico', 'qtd_adequado', 'qtd_avancado',
    'total_alunos', 'pct_abx_basico', 'pct_basico', 'pct_adequado', 'pct_avancado'
]

resumo = df[colunas_numericas].describe().T
resumo['mediana'] = df[colunas_numericas].median()
resumo['moda'] = df[colunas_numericas].mode().iloc[0]
resumo['nulos'] = df[colunas_numericas].isnull().sum()

resumo[['count', 'mean', 'std', 'min', 'mediana', 'moda', '50%', 'max', 'nulos']]

**Análise das estatísticas descritivas:**

- A **proficiência média** geral é de aproximadamente 208 pontos, com desvio-padrão de aprox. 44 pontos, indicando dispersão considerável entre as escolas. O valor mínimo (aprox. 69) e máximo (aprox. 371) reforçam a grande amplitude dos resultados.
- O **total de alunos** por escola/avaliação varia de 1 a 372, com mediana de 30 alunos. Escolas com pouquíssimos alunos avaliados (1-5) podem distorcer os percentuais e merecem atenção no pré-processamento.
- As colunas de percentual (abaixo do básico, básico, adequado, avançado) apresentam médias que, somadas, totalizam 100% — o que indica consistência interna nos dados.
- O percentual médio de alunos no nível **Abaixo do Básico** é de aproximadamente 20%, enquanto o de **Adequado** e **Avançado** somados ficam próximos de 47%, ligeiramente abaixo dos 50% que definimos como limiar de desempenho satisfatório.

### 5.3 Distribuição das variáveis categóricas

Analisamos a frequência das variáveis categóricas para entender a composição do dataset.

In [ ]:
# Distribuição das variáveis categóricas
categoricas = ['NomeDepBol', 'SERIE_ANO', 'COMPONENTE', 'ANO']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribuição das Variáveis Categóricas', fontsize=16, fontweight='bold', y=1.02)

for ax, col in zip(axes.flat, categoricas):
    contagem = df[col].value_counts()
    barras = ax.bar(contagem.index.astype(str), contagem.values, color=CORES['primaria'],
                    edgecolor='white', linewidth=0.8)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_ylabel('Frequência')
    ax.tick_params(axis='x', rotation=15)

    for barra in barras:
        ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 300,
                f'{int(barra.get_height()):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

**Análise das variáveis categóricas:**

- **Rede de ensino (NomeDepBol):** A rede municipal responde por cerca de 61% dos registros, e a estadual por 39%. Isso reflete a estrutura educacional do estado de São Paulo, onde os municípios são responsáveis pelos anos iniciais do Ensino Fundamental.
- **Série/Ano:** O 2º e o 5º ano possuem quantidade semelhante de registros (aprox. 36-37 mil cada), enquanto o 9º ano tem menos (aprox.23 mil). Isso ocorre porque o 9º ano é avaliado predominantemente na rede estadual.
- **Componente curricular:** Distribuição praticamente idêntica entre Língua Portuguesa e Matemática, o que é esperado, pois cada escola avaliada faz prova de ambas as disciplinas.
- **Ano de aplicação:** Distribuição equilibrada entre 2023, 2024 e 2025, com leve crescimento ao longo dos anos.

### 5.4 Distribuição da proficiência

A proficiência é a principal métrica contínua do dataset. Analisamos sua distribuição geral e segmentada por variáveis-chave.

In [ ]:
# Distribuição geral da proficiência
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['PROFIC'], bins=50, color=CORES['primaria'], edgecolor='white',
             alpha=0.85, linewidth=0.5)
axes[0].axvline(df['PROFIC'].mean(), color=CORES['alerta'], linestyle='--',
                linewidth=2, label=f'Média: {df["PROFIC"].mean():.1f}')
axes[0].axvline(df['PROFIC'].median(), color=CORES['sucesso'], linestyle='--',
                linewidth=2, label=f'Mediana: {df["PROFIC"].median():.1f}')
axes[0].set_title('Distribuição da Proficiência (Histograma)', fontweight='bold')
axes[0].set_xlabel('Proficiência')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Boxplot
bp = axes[1].boxplot(df['PROFIC'], vert=True, patch_artist=True,
                     boxprops=dict(facecolor=CORES['primaria'], alpha=0.6),
                     medianprops=dict(color=CORES['alerta'], linewidth=2))
axes[1].set_title('Distribuição da Proficiência (Boxplot)', fontweight='bold')
axes[1].set_ylabel('Proficiência')

plt.tight_layout()
plt.show()

**Análise da distribuição da proficiência:**

- A distribuição é **aproximadamente normal**, com leve assimetria à direita. Média e mediana são muito próximas, confirmando a simetria.
- O boxplot revela a presença de **outliers** em ambas as extremidades, especialmente valores muito baixos (abaixo de 120) e muito altos (acima de 320). Esses casos extremos podem representar escolas com poucos alunos avaliados, onde um ou dois alunos com desempenho atípico afetam fortemente a média da escola.
- A concentração central dos dados (entre Q1 e Q3) ocorre entre aproximadamente 175 e 240 pontos.

In [ ]:
# Proficiência por Rede e Componente
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por rede
sns.boxplot(data=df, x='NomeDepBol', y='PROFIC', palette=[CORES['primaria'], CORES['secundaria']],
            ax=axes[0])
axes[0].set_title('Proficiência por Rede de Ensino', fontweight='bold')
axes[0].set_xlabel('Rede')
axes[0].set_ylabel('Proficiência')

# Por componente
sns.boxplot(data=df, x='COMPONENTE', y='PROFIC', palette=[CORES['sucesso'], CORES['alerta']],
            ax=axes[1])
axes[1].set_title('Proficiência por Componente Curricular', fontweight='bold')
axes[1].set_xlabel('Componente')
axes[1].set_ylabel('Proficiência')

plt.tight_layout()
plt.show()

**Análise da proficiência por segmentos:**

- **Por rede:** A rede estadual apresenta proficiência mediana ligeiramente superior à municipal. Isso pode estar associado ao fato de a rede estadual concentrar o 9º ano, onde as escalas de proficiência são diferentes e os valores tendem a ser maiores em termos absolutos.
- **Por componente:** As distribuições de Língua Portuguesa e Matemática são bastante semelhantes, embora Matemática apresente uma dispersão ligeiramente maior. Isso pode refletir a natureza das habilidades avaliadas, Matemática tende a ter maior variabilidade entre escolas.

In [ ]:
# Proficiência por Série/Ano
fig, ax = plt.subplots(figsize=(10, 5))

ordem_series = ['2º Ano EF', '5º Ano EF', '9º Ano EF']
sns.violinplot(data=df, x='SERIE_ANO', y='PROFIC', order=ordem_series,
               palette=CORES['niveis'][:3], ax=ax, inner='box')
ax.set_title('Distribuição da Proficiência por Série', fontweight='bold', fontsize=14)
ax.set_xlabel('Série / Ano')
ax.set_ylabel('Proficiência')

plt.tight_layout()
plt.show()

**Análise por série:**

- Cada série opera em uma **escala de proficiência distinta**, o que é inerente ao desenho da avaliação SARESP. O 2º ano apresenta valores concentrados na faixa 140-220, o 5º ano entre 160-260, e o 9º ano entre 200-300.
- Os gráficos de violino revelam formas de distribuição diferentes para cada série. O 2º ano apresenta distribuição mais compacta, enquanto o 9º ano é mais disperso.
- **Ponto de atenção:** como as escalas são diferentes entre séries, comparações diretas de proficiência entre séries devem ser feitas com cautela. Os percentuais por nível de desempenho são mais adequados para esse tipo de comparação.

### 5.5 Distribuição dos níveis de desempenho

Analisamos como os alunos se distribuem entre os quatro níveis de desempenho do SARESP.

In [ ]:
# Percentual médio por nível de desempenho
niveis = ['pct_abx_basico', 'pct_basico', 'pct_adequado', 'pct_avancado']
nomes_niveis = ['Abaixo do\nBásico', 'Básico', 'Adequado', 'Avançado']

medias_niveis = [df[col].mean() for col in niveis]

fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.bar(nomes_niveis, medias_niveis, color=CORES['niveis'], edgecolor='white',
                linewidth=1.2, width=0.6)

for barra, media in zip(barras, medias_niveis):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.5,
            f'{media:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Percentual Médio de Alunos por Nível de Desempenho', fontweight='bold', fontsize=14)
ax.set_ylabel('Percentual Médio (%)')
ax.set_ylim(0, max(medias_niveis) + 8)

plt.tight_layout()
plt.show()

**Análise dos níveis de desempenho:**

- O nível **Básico** concentra a maior parcela dos alunos (~33%), seguido por Adequado (~27%), Abaixo do Básico (~20%) e Avançado (~20%).
- Somando os níveis Adequado e Avançado, temos aproximadamente 47% dos alunos com desempenho satisfatório — ligeiramente abaixo do limiar de 50% que definimos.
- Os ~20% de alunos no nível Abaixo do Básico representam um contingente significativo de estudantes que não atingiram sequer as competências mínimas esperadas, o que é um dado preocupante do ponto de vista educacional.

In [ ]:
# Distribuição dos níveis por série e rede (stacked bar)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, agrupador, titulo in zip(axes, ['SERIE_ANO', 'NomeDepBol'],
                                  ['Série/Ano', 'Rede de Ensino']):
    medias = df.groupby(agrupador)[niveis].mean()
    if agrupador == 'SERIE_ANO':
        medias = medias.reindex(ordem_series)

    medias.columns = ['Abaixo do Básico', 'Básico', 'Adequado', 'Avançado']
    medias.plot(kind='barh', stacked=True, color=CORES['niveis'], ax=ax,
                edgecolor='white', linewidth=0.5)
    ax.set_title(f'Níveis de Desempenho por {titulo}', fontweight='bold')
    ax.set_xlabel('Percentual Médio (%)')
    ax.set_ylabel('')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim(0, 105)

plt.tight_layout()
plt.show()

**Análise cruzada dos níveis de desempenho:**

- **Por série:** O 2º ano apresenta o melhor perfil de desempenho, com maior proporção de alunos nos níveis Adequado e Avançado. O 9º ano concentra a maior proporção de alunos no nível Abaixo do Básico, confirmando a hipótese H2 levantada anteriormente.
- **Por rede:** A rede municipal apresenta desempenho ligeiramente superior nos percentuais de Adequado e Avançado quando comparada à rede estadual. Isso pode estar relacionado ao fato de que a rede municipal atende predominantemente os anos iniciais (2º e 5º), que naturalmente apresentam melhores indicadores percentuais.

### 5.6 Distribuição da variável-alvo (classificação)

Construímos a variável-alvo `desempenho_satisfatorio` para verificar o balanceamento das classes.

In [ ]:
# Criar variável-alvo
df['desempenho_satisfatorio'] = (
    (df['pct_adequado'] + df['pct_avancado']) >= 50
).astype(int)

contagem_classes = df['desempenho_satisfatorio'].value_counts()
pct_classes = df['desempenho_satisfatorio'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(['Insatisfatório (0)', 'Satisfatório (1)'], contagem_classes.values,
                color=[CORES['alerta'], CORES['sucesso']], edgecolor='white',
                linewidth=1.2, width=0.5)

for barra, qtd, pct in zip(barras, contagem_classes.values, pct_classes.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 300,
            f'{qtd:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Distribuição da Variável-Alvo', fontweight='bold', fontsize=14)
ax.set_ylabel('Quantidade de Registros')
ax.set_ylim(0, max(contagem_classes.values) * 1.15)

plt.tight_layout()
plt.show()

print(f'Razão entre classes: {contagem_classes.min() / contagem_classes.max():.3f}')

**Análise do balanceamento:**

- A variável-alvo apresenta distribuição **muito bem balanceada**: aproximadamente 52% das escolas são classificadas como insatisfatórias e 48% como satisfatórias.
- A razão entre classes é de ~0.92, o que está bem acima do limiar de 0.5 tipicamente usado para definir desbalanceamento. Isso significa que **não há necessidade de técnicas de balanceamento** (como SMOTE ou undersampling) para futuros modelos de classificação.
- Este resultado é positivo, pois garante que o modelo terá exemplos suficientes de ambas as classes para aprender padrões representativos.

### 5.7 Análise de correlações

Investigamos as relações lineares entre as variáveis numéricas por meio de uma matriz de correlação.

In [ ]:
# Matriz de correlação
colunas_corr = [
    'PROFIC', 'total_alunos', 'pct_abx_basico', 'pct_basico',
    'pct_adequado', 'pct_avancado', 'desempenho_satisfatorio'
]

correlacao = df[colunas_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mascara = np.triu(np.ones_like(correlacao, dtype=bool))

sns.heatmap(correlacao, mask=mascara, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})

ax.set_title('Matriz de Correlação — Variáveis Numéricas', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

**Análise da matriz de correlação:**

- A **proficiência** (PROFIC) apresenta forte correlação positiva com `pct_adequado` e `pct_avancado`, e forte correlação negativa com `pct_abx_basico`. Isso é intuitivo: escolas com maior proficiência média tendem a ter mais alunos nos níveis superiores.
- A variável-alvo (`desempenho_satisfatorio`) apresenta correlação forte com PROFIC (~0.76), o que confirma que a proficiência é um bom preditor do desempenho satisfatório.
- O `total_alunos` apresenta correlação fraca com as demais variáveis, sugerindo que o tamanho da escola não é um fator determinante para o desempenho — embora essa análise mereça aprofundamento com outras técnicas.
- Observa-se **multicolinearidade** entre os percentuais por nível e a proficiência. Em um futuro modelo de ML, será necessário avaliar a remoção de variáveis redundantes para evitar overfitting.

### 5.8 Evolução temporal

Verificamos como o desempenho se comporta ao longo dos três anos avaliados.

In [ ]:
# Evolução do percentual de desempenho satisfatório por ano
evolucao = df.groupby('ANO')['desempenho_satisfatorio'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(evolucao.index, evolucao.values, marker='o', markersize=10, linewidth=2.5,
        color=CORES['primaria'])

for ano, valor in zip(evolucao.index, evolucao.values):
    ax.annotate(f'{valor:.1f}%', (ano, valor), textcoords='offset points',
                xytext=(0, 15), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Evolução do Percentual de Escolas com Desempenho Satisfatório',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Ano da Avaliação')
ax.set_ylabel('% de Escolas Satisfatórias')
ax.set_xticks(evolucao.index)
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

**Análise da evolução temporal:**

- O gráfico mostra a trajetória do percentual de escolas com desempenho satisfatório ao longo dos anos 2023-2025. Variações nesse indicador podem refletir mudanças nas políticas educacionais, nos perfis de alunos avaliados ou na dificuldade das provas.
- É importante lembrar que o SARESP utiliza a Teoria de Resposta ao Item (TRI) para equalização das provas entre anos, o que permite, em tese, comparações longitudinais — embora o indicador percentual usado aqui dependa também da distribuição dos alunos entre níveis.

### 5.9 Análise de escolas com poucos alunos

Escolas com pouquíssimos alunos avaliados podem gerar resultados extremos e distorcer análises.

In [ ]:
# Distribuição do total de alunos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['total_alunos'], bins=60, color=CORES['primaria'], edgecolor='white', alpha=0.85)
axes[0].axvline(10, color=CORES['alerta'], linestyle='--', linewidth=2, label='Limiar: 10 alunos')
axes[0].set_title('Distribuição do Total de Alunos por Avaliação', fontweight='bold')
axes[0].set_xlabel('Total de Alunos')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Proporção abaixo de limiares
limiares = [1, 3, 5, 10, 15, 20]
proporcoes = [(df['total_alunos'] <= lim).mean() * 100 for lim in limiares]

axes[1].bar([str(l) for l in limiares], proporcoes, color=CORES['secundaria'],
            edgecolor='white', linewidth=0.8)

for i, (lim, prop) in enumerate(zip(limiares, proporcoes)):
    axes[1].text(i, prop + 0.3, f'{prop:.1f}%', ha='center', fontsize=10, fontweight='bold')

axes[1].set_title('% de Registros com Total de Alunos ≤ Limiar', fontweight='bold')
axes[1].set_xlabel('Limiar (nº de alunos)')
axes[1].set_ylabel('% dos Registros')

plt.tight_layout()
plt.show()

**Análise de escolas com poucos alunos:**

- Uma parcela dos registros corresponde a escolas com menos de 10 alunos avaliados. Nesses casos, cada aluno tem peso significativo nos percentuais, o que pode levar a valores extremos (ex.: 100% Avançado com apenas 2 alunos).
- Na etapa de pré-processamento, avaliaremos se é necessário aplicar um **filtro mínimo de alunos** para garantir a robustez estatística dos resultados. Um limiar razoável seria de 10 alunos.

### 5.10 Verificação de consistência dos dados

Verificamos se os percentuais por nível somam 100% e se há inconsistências.

In [ ]:
# Verificar se pct_* somam ~100%
soma_pct = df[niveis].sum(axis=1)

print(f'Soma dos percentuais por registro:')
print(f'  Mínimo: {soma_pct.min():.2f}%')
print(f'  Máximo: {soma_pct.max():.2f}%')
print(f'  Média:  {soma_pct.mean():.2f}%')
print(f'  Registros com soma ≠ 100%: {(soma_pct.round(1) != 100.0).sum()}')

# Verificar se qtd_* = total_alunos
soma_qtd = df[['qtd_abx_basico', 'qtd_basico', 'qtd_adequado', 'qtd_avancado']].sum(axis=1)
inconsistentes = (soma_qtd != df['total_alunos']).sum()
print(f'\nRegistros onde soma das qtd ≠ total_alunos: {inconsistentes}')

**Resultado da verificação:**

- Os percentuais somam 100% (ou muito próximo, com diferenças de arredondamento) em todos os registros, confirmando a **consistência interna** do dataset.
- As quantidades por nível somam exatamente o total de alunos em todos os registros.
- Essas verificações são essenciais para garantir a confiabilidade dos dados antes de prosseguir com o pré-processamento.

## 6. Pré-processamento de Dados

Com base nos achados da análise exploratória, realizaremos as seguintes operações de pré-processamento, justificando cada decisão:

1. **Conversão de tipos:** colunas com vírgula decimal e sufixo `%`
2. **Criação da variável-alvo:** classificação binária de desempenho
3. **Filtragem de registros:** remoção de escolas com poucos alunos
4. **Remoção de colunas:** exclusão de identificadores e colunas redundantes
5. **Codificação de categóricas:** Label Encoding e One-Hot Encoding
6. **Normalização e padronização:** MinMaxScaler e StandardScaler
7. **Salvamento de versões:** datasets processados em diferentes formatos

### 6.1 Conversão de tipos de dados

Conforme identificado na EDA, as colunas `PROFIC`, `Produto` e as colunas de percentual estão armazenadas como string. Realizamos a conversão para tipos numéricos adequados.

In [ ]:
# Iniciar com cópia limpa dos dados brutos
df_proc = df_raw.copy()

# 1) Converter colunas com vírgula decimal para float
colunas_virgula_decimal = ['PROFIC', 'Produto']
for col in colunas_virgula_decimal:
    df_proc[col] = df_proc[col].str.replace(',', '.').astype(float)

# 2) Converter colunas de percentual (remover '%', trocar vírgula por ponto)
colunas_percentual = ['pct_abx_basico', 'pct_basico', 'pct_adequado', 'pct_avancado']
for col in colunas_percentual:
    df_proc[col] = df_proc[col].str.replace('%', '').str.replace(',', '.').astype(float)

print('Tipos após conversão:')
print(df_proc[colunas_virgula_decimal + colunas_percentual].dtypes)

### 6.2 Criação da variável-alvo

Criamos a variável binária `desempenho_satisfatorio` com base no critério definido na seção de definição do problema: soma de `pct_adequado` + `pct_avancado` ≥ 50%.

In [ ]:
# Criar variável-alvo
df_proc['desempenho_satisfatorio'] = (
    (df_proc['pct_adequado'] + df_proc['pct_avancado']) >= 50
).astype(int)

print(f'Distribuição da variável-alvo:')
print(df_proc['desempenho_satisfatorio'].value_counts())
print(f'\nProporção: {df_proc["desempenho_satisfatorio"].mean():.2%} satisfatório')

### 6.3 Filtragem de registros com poucos alunos

Removemos registros de escolas com menos de 10 alunos avaliados. Essa decisão se justifica porque escolas com amostras muito pequenas produzem indicadores percentuais instáveis e pouco confiáveis — por exemplo, uma escola com 2 alunos avaliados onde 1 obtém nível Avançado teria 50% de Avançado, o que não é representativo.

In [ ]:
LIMIAR_MIN_ALUNOS = 10

registros_antes = len(df_proc)
df_proc = df_proc[df_proc['total_alunos'] >= LIMIAR_MIN_ALUNOS].copy()
registros_depois = len(df_proc)

print(f'Registros antes da filtragem: {registros_antes:,}')
print(f'Registros removidos:          {registros_antes - registros_depois:,}')
print(f'Registros após filtragem:     {registros_depois:,}')
print(f'Percentual removido:          {(registros_antes - registros_depois) / registros_antes:.2%}')

### 6.4 Seleção de features e remoção de colunas

Removemos colunas que não seriam úteis como features de um modelo de classificação:

- **Identificadores:** `CDREDE`, `CODMUN`, `cod_ibge`, `MUN`, `CODESC`, `NOMESC` — são códigos ou nomes que identificam registros, mas não carregam informação preditiva generalizável.
- **Coluna derivada:** `Produto` — é simplesmente `PROFIC × total_alunos`, redundante.
- **Colunas de quantidade absoluta:** `qtd_abx_basico`, `qtd_basico`, `qtd_adequado`, `qtd_avancado` — a informação já está representada nas colunas de percentual, que são mais comparáveis entre escolas de tamanhos diferentes.

In [ ]:
# Colunas a remover
colunas_remover = [
    'CDREDE', 'CODMUN', 'cod_ibge', 'MUN', 'CODESC', 'NOMESC',
    'Produto', 'qtd_abx_basico', 'qtd_basico', 'qtd_adequado', 'qtd_avancado'
]

df_proc.drop(columns=colunas_remover, inplace=True)

print(f'Colunas restantes ({len(df_proc.columns)}):')
print(list(df_proc.columns))

### 6.5 Codificação de variáveis categóricas

As variáveis categóricas restantes — `NomeDepBol`, `SERIE_ANO` e `COMPONENTE` — precisam ser convertidas para formato numérico. Produziremos duas versões do dataset:

- **Label Encoding:** atribui um inteiro a cada categoria. Útil para algoritmos baseados em árvore (Random Forest, XGBoost).
- **One-Hot Encoding:** cria colunas binárias para cada categoria. Necessário para algoritmos que assumem relação de ordem entre valores numéricos (Regressão Logística, SVM, Redes Neurais).

In [ ]:
# ----- Versão com Label Encoding -----
df_label = df_proc.copy()

colunas_categoricas = ['NomeDepBol', 'SERIE_ANO', 'COMPONENTE']
encoders = {}

for col in colunas_categoricas:
    le = LabelEncoder()
    df_label[col] = le.fit_transform(df_label[col])
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('Mapeamento do Label Encoding:')
for col, mapa in encoders.items():
    print(f'  {col}: {mapa}')

In [ ]:
# ----- Versão com One-Hot Encoding -----
df_onehot = pd.get_dummies(df_proc, columns=colunas_categoricas, drop_first=False,
                            dtype=int)

print(f'Dimensões após One-Hot Encoding: {df_onehot.shape}')
print(f'\nColunas criadas:')
print([col for col in df_onehot.columns if col not in df_proc.columns])

### 6.6 Normalização e padronização

Produzimos versões adicionais do dataset com as variáveis numéricas contínuas normalizadas (MinMaxScaler, escala 0-1) e padronizadas (StandardScaler, média 0 e desvio-padrão 1). Essas transformações são essenciais para algoritmos sensíveis à escala dos dados, como KNN, SVM e Redes Neurais.

**Nota:** A variável-alvo não é transformada, e as variáveis categóricas codificadas com One-Hot Encoding também não, pois já estão em escala binária.

In [ ]:
# Identificar colunas numéricas contínuas (para normalizar/padronizar)
colunas_escalar = ['PROFIC', 'total_alunos', 'pct_abx_basico', 'pct_basico',
                   'pct_adequado', 'pct_avancado']

# ----- Normalização (MinMaxScaler 0-1) -----
df_normalizado = df_onehot.copy()
scaler_minmax = MinMaxScaler()
df_normalizado[colunas_escalar] = scaler_minmax.fit_transform(df_normalizado[colunas_escalar])

print('Dataset normalizado (MinMaxScaler) — amostra:')
df_normalizado[colunas_escalar].describe().loc[['min', 'max', 'mean']]

In [ ]:
# ----- Padronização (StandardScaler) -----
df_padronizado = df_onehot.copy()
scaler_standard = StandardScaler()
df_padronizado[colunas_escalar] = scaler_standard.fit_transform(df_padronizado[colunas_escalar])

print('Dataset padronizado (StandardScaler) — amostra:')
df_padronizado[colunas_escalar].describe().loc[['mean', 'std', 'min', 'max']]

### 6.7 Resumo das versões do dataset

Ao final do pré-processamento, dispomos das seguintes versões do dataset, prontas para diferentes algoritmos de aprendizado de máquina:

| Versão | Encoding | Escala | Uso recomendado |
|---|---|---|---|
| `df_label` | Label Encoding | Original | Árvores de decisão, Random Forest, XGBoost |
| `df_onehot` | One-Hot Encoding | Original | Baseline para qualquer algoritmo |
| `df_normalizado` | One-Hot Encoding | MinMax (0-1) | KNN, Redes Neurais |
| `df_padronizado` | One-Hot Encoding | Standard (z-score) | SVM, Regressão Logística |

In [ ]:
# Resumo das dimensões de cada versão
versoes = {
    'df_label': df_label,
    'df_onehot': df_onehot,
    'df_normalizado': df_normalizado,
    'df_padronizado': df_padronizado
}

for nome, dataset in versoes.items():
    print(f'{nome}: {dataset.shape[0]:,} registros × {dataset.shape[1]} colunas')

## 7. Análise Exploratória Pós-Pré-Processamento

Após as transformações realizadas, revisitamos alguns gráficos utilizando os dados processados para verificar se surgem insights diferentes.

In [ ]:
# Proficiência vs. Desempenho Satisfatório (dados filtrados)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: proficiência vs. pct_adequado + pct_avancado
df_plot = df_onehot.copy()
df_plot['pct_satisfatorio'] = df_plot['pct_adequado'] + df_plot['pct_avancado']

cores_scatter = df_plot['desempenho_satisfatorio'].map({0: CORES['alerta'], 1: CORES['sucesso']})
axes[0].scatter(df_plot['PROFIC'], df_plot['pct_satisfatorio'],
                c=cores_scatter, alpha=0.15, s=8, edgecolors='none')
axes[0].axhline(50, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Limiar 50%')
axes[0].set_title('Proficiência vs. % Satisfatório', fontweight='bold')
axes[0].set_xlabel('Proficiência Média')
axes[0].set_ylabel('% Adequado + Avançado')
axes[0].legend()

# Distribuição da proficiência por classe
for classe, cor, label in [(0, CORES['alerta'], 'Insatisfatório'),
                            (1, CORES['sucesso'], 'Satisfatório')]:
    subset = df_plot[df_plot['desempenho_satisfatorio'] == classe]['PROFIC']
    axes[1].hist(subset, bins=50, alpha=0.6, color=cor, label=label, edgecolor='white')

axes[1].set_title('Distribuição da Proficiência por Classe', fontweight='bold')
axes[1].set_xlabel('Proficiência')
axes[1].set_ylabel('Frequência')
axes[1].legend()

plt.tight_layout()
plt.show()

**Análise pós-pré-processamento:**

- O scatter plot confirma a relação forte entre proficiência e a proporção de alunos nos níveis superiores. A separação entre as classes (vermelho = insatisfatório, verde = satisfatório) é visualmente clara, o que sugere boa separabilidade para modelos de classificação.
- A distribuição de proficiência por classe mostra sobreposição na região central (proficiência entre 180 e 240), onde a classificação é mais ambígua. Essa zona de transição será o principal desafio para um futuro modelo preditivo.
- Com a remoção de escolas com poucos alunos, os outliers extremos diminuíram, resultando em distribuições mais suaves e confiáveis.

In [ ]:
# Verificação final das versões normalizadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalizado
df_normalizado[colunas_escalar].boxplot(ax=axes[0], grid=False)
axes[0].set_title('Boxplot — Dados Normalizados (MinMax)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Padronizado
df_padronizado[colunas_escalar].boxplot(ax=axes[1], grid=False)
axes[1].set_title('Boxplot — Dados Padronizados (Z-score)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

**Análise das transformações de escala:**

- **Normalização (MinMax):** Todas as variáveis foram reescaladas para o intervalo [0, 1]. Isso garante que nenhuma variável domine o cálculo de distâncias em algoritmos como KNN.
- **Padronização (Z-score):** As variáveis agora têm média ~0 e desvio-padrão ~1. Os outliers são mais evidentes nessa escala, aparecendo como pontos extremos nos boxplots. Essa representação é preferida por algoritmos como SVM e Regressão Logística.

## 8. Conclusão

Este trabalho realizou uma análise exploratória e pré-processamento completo dos dados do SARESP, cobrindo as seguintes etapas:

1. **Definição do problema:** Formulei um problema de classificação binária para identificar escolas com desempenho satisfatório (≥50% dos alunos nos níveis Adequado ou Avançado).

2. **Análise exploratória:** Investiguei a distribuição de todas as variáveis, identifiquei padrões por rede de ensino, série e componente curricular, e verifiquei a consistência dos dados.

3. **Principais achados:**
   - O dataset é consistente e não possui valores nulos.
   - A variável-alvo é bem balanceada (aprox. 52% / 48%), dispensando técnicas de balanceamento.
   - A proficiência é um forte preditor do desempenho satisfatório (correlação 0.76).
   - Os anos iniciais (2º ano) apresentam melhores indicadores percentuais que o 9º ano.
   - Escolas com poucos alunos avaliados (<10) geram indicadores instáveis.

4. **Pré-processamento:** Realizei conversão de tipos, filtragem de registros, engenharia de features, codificação de categóricas e transformações de escala, gerando quatro versões do dataset prontas para diferentes famílias de algoritmos de aprendizado de máquina.

O dataset está pronto para a etapa de modelagem, onde poderemos treinar e avaliar modelos de classificação para prever o desempenho escolar no SARESP.

## 9. Exportação para Ingestão em BigQuery

Como etapa final, exportei o dataset pré-processado em formato pronto para ingestão em um data warehouse. Optei por gerar dois formatos:

- **CSV:** formato universal, compatível com qualquer ferramenta. No BigQuery, pode ser carregado via console, `bq load`, ou transferência programática (GCS → BigQuery).
- **Parquet:** formato colunar binário, preferido para pipelines de dados modernos. Preserva os tipos de dados nativamente (sem ambiguidade entre int/float/string), é compactado por padrão e o BigQuery o lê com melhor performance do que CSV.

Exportarei a versão **`df_onehot`** (One-Hot Encoding, escala original), pois é a mais versátil: normalizações e padronizações podem ser aplicadas downstream conforme a necessidade do modelo, enquanto reverter essas transformações exigiria os parâmetros dos scalers.

### Convenções adotadas para compatibilidade com BigQuery

- Nomes de colunas em **snake_case** e sem caracteres especiais (BigQuery não aceita `º`, acentos ou espaços em nomes de coluna).
- Tipos explícitos: inteiros para flags e categóricas codificadas, float para métricas contínuas.
- Sem índice do Pandas no arquivo exportado.

In [ ]:
import re
import unicodedata


def normalizar_nome_coluna(nome: str) -> str:
    """Converte nome de coluna para snake_case compatível com BigQuery.

    Remove acentos, substitui espaços e caracteres especiais por underscore,
    e converte para minúsculas.
    """
    # Remover acentos
    nome = unicodedata.normalize('NFKD', nome).encode('ascii', 'ignore').decode('ascii')
    # Substituir caracteres não alfanuméricos por underscore
    nome = re.sub(r'[^a-zA-Z0-9]', '_', nome)
    # Remover underscores múltiplos e das extremidades
    nome = re.sub(r'_+', '_', nome).strip('_')
    return nome.lower()


# Preparar dataset para exportação
df_export = df_onehot.copy()

# Renomear colunas para snake_case
mapeamento_colunas = {col: normalizar_nome_coluna(col) for col in df_export.columns}
df_export.rename(columns=mapeamento_colunas, inplace=True)

# Exibir mapeamento para rastreabilidade
print('Mapeamento de colunas (original → BigQuery):')
for original, novo in mapeamento_colunas.items():
    marcador = ' ✓' if original != novo else ''
    print(f'  {original:40s} → {novo}{marcador}')

print(f'\nDimensões finais: {df_export.shape[0]:,} × {df_export.shape[1]}')

In [ ]:
# Verificar tipos finais do dataset de exportação
print('Schema do dataset exportado:')
print('-' * 45)
for col in df_export.columns:
    dtype_pandas = df_export[col].dtype
    # Mapear para tipos BigQuery equivalentes
    if dtype_pandas == 'float64':
        bq_type = 'FLOAT64'
    elif dtype_pandas in ('int64', 'int32', 'uint8'):
        bq_type = 'INT64'
    else:
        bq_type = 'STRING'
    print(f'  {col:40s} {str(dtype_pandas):10s} → BQ {bq_type}')

In [ ]:
# ----- Exportação CSV -----
caminho_csv = 'saresp_processado_bigquery.csv'
df_export.to_csv(caminho_csv, index=False, encoding='utf-8')

print(f'CSV exportado: {caminho_csv}')
print(f'  Registros: {len(df_export):,}')

# Verificar tamanho do arquivo
import os
tamanho_mb = os.path.getsize(caminho_csv) / (1024 * 1024)
print(f'  Tamanho:   {tamanho_mb:.1f} MB')

In [ ]:
# ----- Exportação Parquet -----
caminho_parquet = 'saresp_processado_bigquery.parquet'
df_export.to_parquet(caminho_parquet, index=False, engine='pyarrow')

tamanho_parquet_mb = os.path.getsize(caminho_parquet) / (1024 * 1024)

print(f'Parquet exportado: {caminho_parquet}')
print(f'  Registros: {len(df_export):,}')
print(f'  Tamanho:   {tamanho_parquet_mb:.1f} MB')
print(f'\nCompressão Parquet vs CSV: {(1 - tamanho_parquet_mb / tamanho_mb):.0%} menor')

In [ ]:
# Download dos arquivos (no Google Colab)
try:
    from google.colab import files
    files.download(caminho_csv)
    files.download(caminho_parquet)
    print('Downloads iniciados.')
except ImportError:
    print('Ambiente não é Google Colab — arquivos salvos no diretório local.')
    print(f'  → {caminho_csv}')
    print(f'  → {caminho_parquet}')